# Bronze Layer: Raw Data Ingestion

**Purpose:** Load raw CSV files from volumes into bronze Delta tables

**Environment:** Parameterized for dev/uat/prod deployment

**CI/CD Ready:** Uses environment parameter for multi-environment execution

In [0]:
# Get environment parameter (set via widget or job parameter)
env = dbutils.widgets.get("environment")

# Set catalog name based on environment
catalog = f"1_{env}"

# Set volume path (adjust if different per environment)
volume_path = f"/Volumes/{catalog}/bronze/source_files/"

# Print configuration for verification
print(f"="*60)
print(f"BRONZE LAYER CONFIGURATION")
print(f"="*60)
print(f"Environment: {env}")
print(f"Catalog: {catalog}")
print(f"Volume Path: {volume_path}")
print(f"="*60)

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS 1_dev")
spark.sql("CREATE CATALOG IF NOT EXISTS 1_uat")
spark.sql("CREATE CATALOG IF NOT EXISTS 1_prod")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_dev.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_dev.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_dev.gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_uat.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_uat.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_uat.gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_prod.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_prod.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS 1_prod.gold")

In [0]:
# Define source files
files = ["customers.csv", "order_items.csv", "orders.csv", "products.csv"]

# Load each CSV file into its corresponding bronze table
for file in files:
    table_name = file.replace(".csv", "")
    file_path = f"{volume_path}{file}"
    
    # Read CSV with header and infer schema
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    # Add ingestion metadata
    from pyspark.sql.functions import current_timestamp, lit
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file_name", lit(file))
    
    # Write to bronze table in environment-specific catalog
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.bronze.{table_name}")
    
    print(f"✓ Loaded {file} to {catalog}.bronze.{table_name}")

print(f"\n✓ All files loaded successfully to {catalog}.bronze!")